# Week 1 · Exercise 1: pandas, NumPy, pipelines and cross-validation

**Goal:** predict which loan applicants default, and do it without fooling yourself.

Data: `make_credit_applicants()` from the `classical_ml` package, a synthetic set of 5,000 loan
applicants (~12% default). It is generated from a known formula, so we know what the "right"
answer looks like. Next week you swap in a real dataset.

The finished, tested code already lives in `classical_ml/src/classical_ml/`. This notebook is the
scratchpad where you see *why* each piece is there. Look for **✏️ Try it** cells.

## 1. pandas tour: look before you model

In [1]:
import numpy as np
import pandas as pd

from classical_ml.datasets import CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGET, make_credit_applicants

df = make_credit_applicants()
df.head()

,age,annual_income,loan_amount,credit_history_months,num_late_payments,employment_type,region,owns_home,defaulted
0,63,137000.0,76200.0,391.0,1,salaried,south,no,0
1,52,60500.0,56700.0,349.0,1,salaried,east,no,0
2,46,53900.0,45600.0,93.0,1,self_employed,north,yes,0
3,34,61100.0,70500.0,140.0,2,salaried,south,no,0
4,36,63000.0,17300.0,58.0,0,salaried,south,no,0


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   age                    5000 non-null   int64  
 1   annual_income          4742 non-null   float64
 2   loan_amount            5000 non-null   float64
 3   credit_history_months  4765 non-null   float64
 4   num_late_payments      5000 non-null   int64  
 5   employment_type        4764 non-null   str    
 6   region                 5000 non-null   str    
 7   owns_home              5000 non-null   str    
 8   defaulted              5000 non-null   int64  
dtypes: float64(3), int64(3), str(3)
memory usage: 351.7 KB


In [3]:
df.describe().round(1)

,age,annual_income,loan_amount,credit_history_months,num_late_payments,defaulted
count,5000.0,4742.0,5000.0,4765.0,5000.0,5000.0
mean,45.6,68390.7,45025.1,199.7,0.8,0.1
std,14.6,36387.8,34728.4,137.4,0.9,0.3
min,21.0,11100.0,2200.0,7.0,0.0,0.0
25%,33.0,43000.0,20400.0,88.0,0.0,0.0
50%,46.0,59500.0,36500.0,169.0,1.0,0.0
75%,58.0,84600.0,59500.0,287.0,1.0,0.0
max,70.0,305200.0,323100.0,614.0,6.0,1.0


Missing values per column. Any model you fit has to cope with these.

In [4]:
df.isna().mean().round(3)

age                      0.000
annual_income            0.052
loan_amount              0.000
credit_history_months    0.047
num_late_payments        0.000
employment_type          0.047
region                   0.000
owns_home                0.000
defaulted                0.000
dtype: float64

`groupby` is the workhorse. Default rate by employment type: does it match intuition?
(`dropna=False` keeps the rows where employment type is missing as their own group.)

In [5]:
df.groupby("employment_type", dropna=False)[TARGET].agg(["mean", "count"]).sort_values("mean")

,mean,count
employment_type,,
salaried,0.092676,2881
NaN,0.131356,236
self_employed,0.138264,933
contract,0.165969,717
unemployed,0.309013,233


**✏️ Try it:** compute the default rate by `region` and by `owns_home`. One of these should
look like noise (the generator ignores it). Which one?

In [8]:
for col in ["region", "owns_home"]:
    print(df.groupby(col)[TARGET].agg(["mean", "count"]), "\n")

            mean  count
region                 
east    0.108865   1286
north   0.133597   1265
south   0.126511   1241
west    0.125828   1208 

               mean  count
owns_home                 
no         0.130864   2835
yes        0.114088   2165 



## 2. NumPy: vectorise, don't loop

Debt-to-income (loan ÷ income) is a classic credit feature. Compare a Python loop with one
vectorised expression.

In [9]:
loan = df["loan_amount"].to_numpy()
income = df["annual_income"].to_numpy()

def dti_loop(loan, income):
    out = []
    for l, i in zip(loan, income):
        out.append(l / i)
    return np.array(out)

%timeit dti_loop(loan, income)
%timeit loan / income
np.allclose(dti_loop(loan, income), loan / income, equal_nan=True)

714 μs ± 5.62 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
2.15 μs ± 18.3 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


True

Same answer, roughly 100× faster. NumPy runs the loop in C over a contiguous array. Note that
NaN income gives NaN debt-to-income: missing values propagate.

**✏️ Try it:** add a `debt_to_income` column to `df` and check its default rate by quartile
with `pd.qcut(df["debt_to_income"], 4)`.

In [10]:
df["history_years"] = df["credit_history_months"] / 12        # new column from two existing ones
band = pd.qcut(df["history_years"], 4)                        # 4 equal-sized buckets
df.groupby(band)[TARGET].agg(["mean", "count"])               # same groupby as cell 8, grouped by the buckets

,mean,count
history_years,,
"(0.582, 7.333]",0.180833,1200
"(7.333, 14.083]",0.130763,1193
"(14.083, 23.917]",0.122792,1189
"(23.917, 51.167]",0.059172,1183


In [11]:
df["debt_to_income"] = df["loan_amount"] / df["annual_income"]
df.groupby(pd.qcut(df["debt_to_income"], 4))[TARGET].agg(["mean", "count"])

,mean,count
debt_to_income,,
"(0.099, 0.387]",0.054806,1186
"(0.387, 0.659]",0.075949,1185
"(0.659, 0.93]",0.144304,1185
"(0.93, 1.2]",0.220911,1186


## 3. Supervised vs unsupervised

- **Supervised:** you have labels (`defaulted`) and learn a mapping features → label.
- **Unsupervised:** no labels. You look for structure, e.g. groups of similar applicants.

Same data, two questions. First, supervised: logistic regression predicting default.

In [12]:
from sklearn.linear_model import LogisticRegression

from classical_ml.pipeline import build_preprocessor

X = df.drop(columns=TARGET)
y = df[TARGET]

X_matrix = build_preprocessor().fit_transform(X)   # numbers only, no NaNs (see section 5)
clf = LogisticRegression(max_iter=1000).fit(X_matrix, y)
clf.predict_proba(X_matrix[:5])[:, 1].round(3)    # P(default) for the first 5 applicants

array([0.025, 0.061, 0.205, 0.293, 0.05 ])

Unsupervised: k-means puts applicants into 4 clusters *without ever seeing `defaulted`*.

In [13]:
from sklearn.cluster import KMeans

clusters = KMeans(n_clusters=4, n_init="auto", random_state=0).fit_predict(X_matrix)
df.assign(cluster=clusters).groupby("cluster")[TARGET].agg(["mean", "count"])

,mean,count
cluster,,
0,0.169617,678
1,0.063299,1643
2,0.116830,1943
3,0.233696,736


If cluster default rates differ, the clusters found real structure that happens to relate to
risk. That is a common use of unsupervised learning: segmentation, then checking what segments mean.

(Yes, section 3 fit the preprocessor on *all* rows. Fine for a demo, a bug for evaluation. That's next.)

## 4. Train / validation / test, and leakage

- **Train:** fit the model.
- **Validation:** compare models and tune settings. You'll look at it many times.
- **Test:** touch it **once**, at the end, to report a number you didn't optimise against.

The repo already has a stratified splitter (keeps the 12% default rate in every split):

In [14]:
from classical_ml.data import train_val_test_split

train, val, test = train_val_test_split(df, target=TARGET)
pd.DataFrame({s: [len(d), d[TARGET].mean()] for s, d in [("train", train), ("val", val), ("test", test)]},
             index=["rows", "default rate"]).round(3)

,train,val,test
rows,3500.000,750.000,750.000
default rate,0.124,0.124,0.123


**Leakage** = information from the evaluation data sneaking into training, so your score is
better than reality. The nastiest kind hides in preprocessing.

Demo: 200 rows of **pure random noise**, 5,000 features, random labels. No model can beat 50%.

In [15]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
X_noise = rng.normal(size=(200, 5000))
y_noise = rng.integers(0, 2, size=200)

# WRONG: pick the 20 "best" features using ALL rows, then cross-validate
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LogisticRegression(), X_selected, y_noise, cv=5).mean()

# RIGHT: feature selection inside the pipeline, so it's refit on each training fold only
honest = cross_val_score(
    make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression()), X_noise, y_noise, cv=5
).mean()

print(f"leaky accuracy:  {leaky:.2f}")
print(f"honest accuracy: {honest:.2f}")

leaky accuracy:  0.79
honest accuracy: 0.51


The leaky version "finds signal" in noise because the feature selector already saw the
validation folds' labels. **Rule: anything that learns from data goes inside the pipeline.**
That includes imputers (they learn medians) and scalers (they learn means).

## 5. `Pipeline` + `ColumnTransformer`

`build_preprocessor()` (in `classical_ml/pipeline.py`) routes columns by type:

| columns | steps |
|---|---|
| numeric | median imputation → standard scaling |
| categorical | most-frequent imputation → one-hot encoding |

`build_pipeline()` puts a model after it. Fitting the pipeline fits every step on the training
data only; predicting just applies them.

In [16]:
from classical_ml.pipeline import build_pipeline

pipe = build_pipeline()
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the o

In [17]:
pipe.fit(train.drop(columns=TARGET), train[TARGET])
pipe.named_steps["preprocess"].get_feature_names_out()

array(['num__age', 'num__annual_income', 'num__loan_amount',
       'num__credit_history_months', 'num__num_late_payments',
       'cat__employment_type_contract', 'cat__employment_type_salaried',
       'cat__employment_type_self_employed',
       'cat__employment_type_unemployed', 'cat__region_east',
       'cat__region_north', 'cat__region_south', 'cat__region_west',
       'cat__owns_home_no', 'cat__owns_home_yes'], dtype=object)

## 6. k-fold cross-validation

One validation split gives one noisy number. 5-fold CV trains 5 times, each time holding out a
different fifth, and gives you a mean ± spread. Stratified so each fold keeps the 12% default rate.

Metric: **ROC-AUC**, the probability the model ranks a random defaulter above a random
non-defaulter (0.5 = coin flip, 1.0 = perfect). Accuracy is useless here: predicting "nobody
defaults" scores 88%. Week 2 goes deep on metrics.

In [18]:
from sklearn.dummy import DummyClassifier

from classical_ml.pipeline import cross_validate

train_val = pd.concat([train, val])
X_tv, y_tv = train_val.drop(columns=TARGET), train_val[TARGET]

print("dummy:   ", cross_validate(build_pipeline(DummyClassifier()), X_tv, y_tv))
print("logistic:", cross_validate(build_pipeline(), X_tv, y_tv))

dummy:    0.500 ± 0.000 over 5 folds
logistic: 0.751 ± 0.034 over 5 folds


Always compare with a dummy baseline. "0.75 AUC" means nothing until you know chance is 0.5.

Finally, **once**, the test set:

In [19]:
from sklearn.metrics import roc_auc_score

final = build_pipeline().fit(X_tv, y_tv)
test_auc = roc_auc_score(test[TARGET], final.predict_proba(test.drop(columns=TARGET))[:, 1])
print(f"test ROC-AUC: {test_auc:.3f}")

test ROC-AUC: 0.742


## ✏️ Your turn

Do these here first, then move anything reusable into the package **with a test** (that's the
Week 1 deliverable).

1. **Model swap.** Pass `RandomForestClassifier(random_state=0)` to `build_pipeline()` and CV it.
   Does it beat logistic regression? Is the gap bigger than the ± spread?
2. **Feature engineering.** The generator uses debt-to-income, but the model only sees loan and
   income separately. Add a `debt_to_income` feature (hint: a `FunctionTransformer` in the
   numeric branch, or a new column in the dataset). How much does CV AUC move?
3. **Drop the noise.** Remove `region` from the categorical features. Does the score change?
4. **Break it on purpose.** Fit `StandardScaler` on the full `X_tv` *before* CV and compare.
   Why is the effect tiny here but huge in the noise demo? (Hint: what does each step learn?)
5. **Look inside.** After fitting `final`, read `final.named_steps["model"].coef_` next to the
   feature names. Do the signs match the generator in `datasets.py`?
6. **Write-up.** Start `notes/week-01.md`: what you built, one number that moved, what confused
   you, one interview question you can now answer (try: *"What is data leakage and how do you
   prevent it?"*).

In [22]:
from sklearn.ensemble import RandomForestClassifier

from classical_ml.pipeline import build_preprocessor

X = df.drop(columns=TARGET)
y = df[TARGET]

X_matrix = build_preprocessor().fit_transform(X)   # numbers only, no NaNs (see section 5)
clf = RandomForestClassifier(random_state=0).fit(X_matrix, y)
clf.predict_proba(X_matrix[:5])[:, 1].round(3)    # P(default) for the first 5 applicants

array([0.  , 0.01, 0.07, 0.19, 0.  ])

In [28]:
from sklearn.pipeline import Pipeline

X_tv2 = X_tv.assign(debt_to_income=X_tv["loan_amount"] / X_tv["annual_income"])

pipe2 = Pipeline([
    ("preprocess", build_preprocessor(numeric=[*NUMERIC_FEATURES, "debt_to_income"])),
    ("model", LogisticRegression(max_iter=1000)),
])
print("with dti:", cross_validate(pipe2, X_tv2, y_tv))

with dti: 0.762 ± 0.032 over 5 folds


In [29]:
no_region = [c for c in CATEGORICAL_FEATURES if c != "region"]

pipe3 = Pipeline([
    ("preprocess", build_preprocessor(categorical=no_region)),
    ("model", LogisticRegression(max_iter=1000)),
])
print("no region:", cross_validate(pipe3, X_tv, y_tv))
print("baseline: ", cross_validate(build_pipeline(), X_tv, y_tv))

no region: 0.749 ± 0.035 over 5 folds
baseline:  0.751 ± 0.034 over 5 folds


In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num = list(NUMERIC_FEATURES)

def make_pipe(scaler_inside):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scaler_inside:
        steps.append(("scale", StandardScaler()))
    return Pipeline([
        ("pre", ColumnTransformer([("num", Pipeline(steps), num)])),
        ("model", LogisticRegression(max_iter=1000)),
    ])

# HONEST: the scaler lives in the pipeline, so each fold fits it on its own training rows
print("honest:", cross_validate(make_pipe(scaler_inside=True), X_tv, y_tv))

# LEAKY: scale ALL rows first (including future validation rows), then cross-validate
X_leaky = X_tv.copy()
X_leaky[num] = StandardScaler().fit_transform(
    SimpleImputer(strategy="median").fit_transform(X_tv[num])
)
print("leaky: ", cross_validate(make_pipe(scaler_inside=False), X_leaky, y_tv))

honest: 0.729 ± 0.029 over 5 folds
leaky:  0.730 ± 0.029 over 5 folds


In [31]:
names = final.named_steps["preprocess"].get_feature_names_out()
coefs = final.named_steps["model"].coef_[0]
pd.Series(coefs, index=names).round(2).sort_values()

cat__employment_type_salaried        -0.67
num__annual_income                   -0.56
num__credit_history_months           -0.55
cat__region_east                     -0.23
cat__employment_type_self_employed   -0.13
cat__owns_home_yes                   -0.12
num__age                             -0.00
cat__region_south                     0.03
cat__employment_type_contract         0.04
cat__region_north                     0.09
cat__owns_home_no                     0.12
cat__region_west                      0.12
num__num_late_payments                0.50
num__loan_amount                      0.76
cat__employment_type_unemployed       0.76
dtype: float64